## Import các thư viện cần thiết

In [1]:
from  datetime import datetime, timedelta
import numpy as np, pandas as pd
import lightgbm as lgb

#Hiển thị max 50 cột
pd.options.display.max_columns = 50

# Trỏ đường dẫn để gọi file trong thư mục src
import sys, os
sys.path.append(os.path.abspath('../../src'))

#Import các hàm đã được viết trong src
from dataPreprocessing import create_dt, create_fea
from featureEngineering import add_demandFeature
from evaluation import rmsse
from inventory_simulation import simulate_periodic_review_L0, simulate_periodic_withForecast_L0

## Định nghĩa các biến cần thiết

In [2]:
#Định nghĩa dữ liệu cho từng cột phân loại
CAL_DTYPES={"event_name_1": "category", "event_name_2": "category", "event_type_1": "category", 
         "event_type_2": "category", "weekday": "category", 'wm_yr_wk': 'int16', "wday": "int16",
        "month": "int16", "year": "int16", "snap_CA": "float32", 'snap_TX': 'float32', 'snap_WI': 'float32' }

#Định nghĩa dữ liệu cho câc cột lien quan đến giá
PRICE_DTYPES = {"store_id": "category", "item_id": "category", "wm_yr_wk": "int16","sell_price":"float32" }

#Ngày bắt đầu lấy dữ liệu
FIRST_DAY = 350

#Số ngày dự báo
h = 28 

#Số lượng lags tối đa
max_lags = 57

#Ngày kết thúc tập huấn luyện
tr_last = 1913

#Ngày đầu tiên của dự báo
fday = datetime(2016,4, 25) 

#Các cột không sử dụng
useless_cols = ["id", "date", "sales","d", "wm_yr_wk", "weekday"]

## Dự báo dữ liệu M5 sử dụng LightGBM

Trong nghiên cứu này, mô hình Light Gradient Boosting Machine (LightGBM) được sử dụng để dự báo nhu cầu trong bộ dữ liệu M5. LightGBM là một thuật toán học máy dựa trên cây quyết định, có khả năng xử lý hiệu quả các tập dữ liệu lớn và phù hợp với bài toán chuỗi thời gian.

### Phương pháp thực hiện

- **Chuẩn bị dữ liệu**:  
  Dữ liệu được chuyển đổi sang dạng supervised learning thông qua việc xây dựng các đặc trưng trễ (lag features) và thống kê trượt (rolling statistics).

- **Xây dựng đặc trưng (Feature Engineering)**:  
  Các đặc trưng chính bao gồm:
  - Đặc trưng trễ: `lag_7`, `lag_28`
  - Thống kê trượt: `rolling_mean_7`, `rolling_std_28`
  - Đặc trưng lịch: ngày trong tuần, tháng, sự kiện đặc biệt
  - Đặc trưng liên quan đến giá bán

- **Huấn luyện mô hình**:  
  LightGBM được huấn luyện với hàm mục tiêu Poisson nhằm mô hình hóa tốt hơn dữ liệu dạng đếm (count data) như nhu cầu.

- **Dự báo lặp (Recursive Forecasting)**:  
  Dự báo được thực hiện theo từng bước trong 28 ngày, trong đó giá trị dự báo ở bước trước được sử dụng làm đầu vào cho bước tiếp theo.

- **Đánh giá mô hình**:  
  Hiệu quả mô hình được đánh giá bằng chỉ số Root Mean Squared Scaled Error (RMSSE), là metric chính thức của cuộc thi M5.

In [42]:
#Load từ mô hình đã được train từ trước
m_lgb = lgb.Booster(model_file='../../dataset/result/model_20260327_085207(best).lgb')

train_cols = ['item_id', 'dept_id', 'store_id', 'cat_id', 'state_id', 'wday', 'month',
       'year', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2',
       'snap_CA', 'snap_TX', 'snap_WI', 'sell_price',
       'cat_Erratic (Thất thường)', 'cat_Intermittent (Ngắt quãng)',
       'cat_Lumpy (Cục bộ)', 'cat_Smooth (Đều đặn)', 'lag_7', 'lag_28',
       'rmean_7_7', 'rmean_28_7', 'rmean_7_28', 'rmean_28_28', 'week',
       'quarter', 'mday']

In [43]:
#Recursive forecasting và magic multiplier
alphas = [1.028, 1.023, 1.018]
weights = [1/len(alphas)]*len(alphas)
sub = 0.

for icount, (alpha, weight) in enumerate(zip(alphas, weights)):

    #Lấy dữ liệu
    te = create_dt(raw_folder= "../../dataset/raw/" ,is_train=False, price_dt = PRICE_DTYPES, cal_dt = CAL_DTYPES, tr_last = tr_last)

    #Add demand feature vào te
    te = add_demandFeature(demand_dir="../../dataset/result/demand_classification.csv", data = te)
    cols = [f"d_{tr_last + i}" for i in range(1,29)]

    for tdelta in range(0, 28):
        day = fday + timedelta(days=tdelta)
        print(tdelta, day)
        tst = te[(te.date >= day - timedelta(days=max_lags)) & (te.date <= day)].copy()

        create_fea(tst ,lags_fe = [7, 28], wins_fe = [7, 28])
        
        tst = tst.loc[tst.date == day , train_cols]
        te.loc[te.date == day, "sales"] = alpha*m_lgb.predict(tst) # magic multiplier by kyakovlev



    te_sub = te.loc[te.date >= fday, ["id", "sales"]].copy()

    te_sub["F"] = [f"d_{tr_last + rank}" 
               for rank in te_sub.groupby("id")["id"].cumcount() + 1]
    te_sub = te_sub.set_index(["id", "F" ]).unstack()["sales"][cols].reset_index()
    te_sub.fillna(0., inplace = True)
    te_sub.sort_values("id", inplace = True)
    te_sub.reset_index(drop=True, inplace = True)
    te_sub.to_csv(f"../../dataset/result/all/submission_all_{icount}.csv",index=False)
    if icount == 0 :
        sub = te_sub
        sub[cols] *= weight
    else:
        sub[cols] += te_sub[cols]*weight
    print(icount, alpha, weight)


0 2016-04-25 00:00:00
1 2016-04-26 00:00:00
2 2016-04-27 00:00:00
3 2016-04-28 00:00:00
4 2016-04-29 00:00:00
5 2016-04-30 00:00:00
6 2016-05-01 00:00:00
7 2016-05-02 00:00:00
8 2016-05-03 00:00:00
9 2016-05-04 00:00:00
10 2016-05-05 00:00:00
11 2016-05-06 00:00:00
12 2016-05-07 00:00:00
13 2016-05-08 00:00:00
14 2016-05-09 00:00:00
15 2016-05-10 00:00:00
16 2016-05-11 00:00:00
17 2016-05-12 00:00:00
18 2016-05-13 00:00:00
19 2016-05-14 00:00:00
20 2016-05-15 00:00:00
21 2016-05-16 00:00:00
22 2016-05-17 00:00:00
23 2016-05-18 00:00:00
24 2016-05-19 00:00:00
25 2016-05-20 00:00:00
26 2016-05-21 00:00:00
27 2016-05-22 00:00:00
0 1.028 0.3333333333333333
0 2016-04-25 00:00:00
1 2016-04-26 00:00:00
2 2016-04-27 00:00:00
3 2016-04-28 00:00:00
4 2016-04-29 00:00:00
5 2016-04-30 00:00:00
6 2016-05-01 00:00:00
7 2016-05-02 00:00:00
8 2016-05-03 00:00:00
9 2016-05-04 00:00:00
10 2016-05-05 00:00:00
11 2016-05-06 00:00:00
12 2016-05-07 00:00:00
13 2016-05-08 00:00:00
14 2016-05-09 00:00:00
15 2

In [44]:
#Lưu dữ liệu forecast
sub["id"] = sub["id"].str.extract(r"^(.+?)_(?:validation|evaluation)$", expand=False)
sub.to_csv("../../dataset/result/all/submission_all.csv")

In [45]:
#Load dữ liệu forecast
sub = pd.read_csv("../../dataset/result/all/submission_all.csv")
sub.set_index("id", inplace = True)
sub

,Unnamed: 0,d_1914,d_1915,d_1916,d_1917,d_1918,d_1919,d_1920,d_1921,d_1922,d_1923,d_1924,d_1925,d_1926,d_1927,d_1928,d_1929,d_1930,d_1931,d_1932,d_1933,d_1934,d_1935,d_1936,d_1937,d_1938,d_1939,d_1940,d_1941
id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
FOODS_1_001_CA_1,0,0.804077,0.792265,0.763281,0.764405,0.983937,1.083061,1.198029,1.033757,0.977757,0.905416,0.900425,1.065092,1.291474,1.145303,0.990333,0.930154,0.918564,0.931144,1.065671,1.328108,1.270246,0.927586,0.853912,0.801496,0.827293,0.966793,1.172268,1.193435
FOODS_1_001_CA_2,1,0.926463,1.006068,0.896102,1.038163,1.168146,1.352504,1.460816,0.946770,0.972879,0.967815,1.003869,1.190947,1.484112,1.314712,1.020276,1.010801,1.028198,1.052107,1.230996,1.559724,1.622909,1.053140,0.987467,1.011320,0.998877,1.170529,1.569784,1.440060
FOODS_1_001_CA_3,2,1.113022,1.065720,0.895490,0.852039,0.900217,1.158423,1.206532,1.188379,1.183703,1.009581,0.996941,1.080790,1.400160,1.212179,1.181112,1.134408,0.993458,0.979787,1.007855,1.605193,1.837181,1.151772,1.087793,0.894128,0.863079,0.936163,1.247931,1.391099
FOODS_1_001_CA_4,3,0.448898,0.348885,0.355739,0.360381,0.422065,0.419693,0.565709,0.409560,0.422543,0.434994,0.447065,0.434807,0.476821,0.384587,0.374756,0.395204,0.399459,0.425015,0.469094,0.510492,0.480327,0.384232,0.363705,0.358890,0.380200,0.419386,0.484549,0.466431
FOODS_1_001_TX_1,4,0.184206,0.181095,0.192601,0.193988,0.168686,0.160562,0.208287,0.532737,0.483866,0.515773,0.485304,0.532747,0.553098,0.444146,0.466805,0.456792,0.397093,0.403119,0.422233,0.386452,0.430684,0.287959,0.286163,0.274292,0.289301,0.314593,0.335013,0.359325
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
HOUSEHOLD_2_516_TX_2,30485,0.227734,0.210416,0.214775,0.224323,0.327861,0.388223,0.341297,0.239947,0.233167,0.226952,0.321924,0.357339,0.366089,0.340309,0.240998,0.243723,0.226862,0.240676,0.296563,0.384431,0.368952,0.265811,0.236578,0.236942,0.246151,0.311480,0.383324,0.369376
HOUSEHOLD_2_516_TX_3,30486,0.164629,0.152799,0.164414,0.155874,0.181803,0.207453,0.172299,0.128729,0.121220,0.109702,0.138463,0.148288,0.189369,0.154933,0.133945,0.121317,0.127896,0.133714,0.163248,0.187414,0.175037,0.141174,0.132232,0.139516,0.146736,0.180510,0.206483,0.194666
HOUSEHOLD_2_516_WI_1,30487,0.088787,0.085345,0.085930,0.093005,0.103318,0.117433,0.107637,0.095018,0.093741,0.090558,0.108157,0.134549,0.156483,0.123726,0.094827,0.090514,0.091272,0.096067,0.129423,0.145548,0.139682,0.097690,0.094000,0.093898,0.099325,0.138210,0.157171,0.149506


In [46]:
#Xử lý lại tập test để đánh giá
df_valid = pd.read_csv("../../dataset/raw/sales_train_evaluation.csv")
cols = ["id"] + [f"d_{i}" for i in range(tr_last + 1, 1942)]
test = df_valid[cols]
test["id"] = test["id"].str.extract(r"^(.+?)_(?:validation|evaluation)$", expand=False)

test.set_index("id", inplace = True)

#Xử lý lại tập train để đánh giá
df_valid = pd.read_csv("../../dataset/raw/sales_train_validation.csv")
df_valid["id"] = df_valid["id"].str.extract(r"^(.+?)_(?:validation|evaluation)$", expand=False)

df_valid.set_index("id", inplace = True)

# 1. Đổi đuôi id
train = df_valid.filter(like="d_")
train.head()


#Load dữ liệu tỷ trọng
item_share = pd.read_csv("../../dataset/result/item_share.csv")
item_share.set_index("item_id", inplace = True)

In [47]:
rmsse_result = rmsse(train, test, sub)
print(rmsse_result.mean())

0.7508178854328138


In [48]:
wrmsse = rmsse_result * item_share['sales_share']
wrmsse.sum()

np.float64(0.7425509510936255)

## Đo lường trên phương pháp tồn kho truyền thống

In [3]:
test_long = create_dt(raw_folder= "../../dataset/raw/" , 
                      sales_data = 'sales_train_evaluation.csv', 
                      price_dt = PRICE_DTYPES, 
                      cal_dt = CAL_DTYPES, tr_last = 1941, first_day = 1884)

In [4]:
test_long.head()

,id,item_id,dept_id,store_id,cat_id,state_id,d,sales,date,wm_yr_wk,weekday,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price
0,HOBBIES_1_001_CA_1_evaluation,0,0,0,0,0,d_1884,1.0,2016-03-26,11609,2,1,3,2016,0,0,0,0,0.0,0.0,0.0,8.26
1,HOBBIES_1_002_CA_1_evaluation,1,0,0,0,0,d_1884,1.0,2016-03-26,11609,2,1,3,2016,0,0,0,0,0.0,0.0,0.0,3.97
2,HOBBIES_1_003_CA_1_evaluation,2,0,0,0,0,d_1884,1.0,2016-03-26,11609,2,1,3,2016,0,0,0,0,0.0,0.0,0.0,2.97
3,HOBBIES_1_004_CA_1_evaluation,3,0,0,0,0,d_1884,6.0,2016-03-26,11609,2,1,3,2016,0,0,0,0,0.0,0.0,0.0,4.64
4,HOBBIES_1_005_CA_1_evaluation,4,0,0,0,0,d_1884,0.0,2016-03-26,11609,2,1,3,2016,0,0,0,0,0.0,0.0,0.0,2.88


In [5]:
import pandas as pd
import numpy as np
from scipy.stats import norm

# =========================
# 1. Chuẩn bị dữ liệu
# =========================
df_sim = test_long.copy()

# Tách số ngày từ cột d, ví dụ d_1884 -> 1884
df_sim["d_num"] = df_sim["d"].str.extract(r"(\d+)").astype(int)

# Sắp xếp cho chắc chắn
df_sim = df_sim.sort_values(["id", "d_num"]).reset_index(drop=True)

# =========================
# 2. Tham số mô phỏng
# =========================
review_period = 7
service_level = 0.95

# Giai đoạn ước lượng mu_demand và std_demand
estimation_start = 1884
estimation_end = 1913

# Giai đoạn mô phỏng: từ sau estimation window đến hết dữ liệu
simulation_start = 1914

# =========================
# 3. Nơi lưu kết quả
# =========================
summary_list = []
details_dict = {}

# =========================
# 4. Vòng lặp theo từng sản phẩm
# =========================
for product_id, group in df_sim.groupby("id"):
    group = group.sort_values("d_num").copy()

    # -------------------------
    # 4.1. Dữ liệu estimation
    # -------------------------
    estimation_data = group[
        (group["d_num"] >= estimation_start) & (group["d_num"] <= estimation_end)
    ].copy()

    if estimation_data.empty:
        continue

    mu_demand = estimation_data["sales"].mean()
    std_demand = estimation_data["sales"].std(ddof=1)

    # Nếu std bị NaN do quá ít điểm dữ liệu thì thay bằng 0
    if pd.isna(std_demand):
        std_demand = 0.0

    # -------------------------
    # 4.2. Xác định giá sản phẩm
    # holding cost = 1% giá trị sản phẩm
    # shortage cost = 25% giá trị sản phẩm
    # -------------------------
    price_series = estimation_data["sell_price"].dropna()

    if price_series.empty:
        price_series = group["sell_price"].dropna()

    if price_series.empty:
        continue

    unit_price = price_series.iloc[-1]

    holding_cost = 0.01 * unit_price
    shortage_cost = 0.25 * unit_price

    # -------------------------
    # 4.3. Initial inventory = order up to level
    # dùng đúng logic policy của hàm bạn
    # -------------------------
    z = norm.ppf(service_level)
    order_up_to_level = mu_demand * review_period + z * std_demand * np.sqrt(review_period)
    initial_inventory = order_up_to_level

    # -------------------------
    # 4.4. Demand series để mô phỏng
    # -------------------------
    simulation_data = group[group["d_num"] >= simulation_start].copy()

    if simulation_data.empty:
        continue

    demand_series = simulation_data["sales"].values

    # -------------------------
    # 4.5. Chạy hàm mô phỏng GIỮ NGUYÊN
    # -------------------------
    results_df, summary = simulate_periodic_review_L0(
        mu_demand=mu_demand,
        std_demand=std_demand,
        review_period=review_period,
        service_level=service_level,
        initial_inventory=initial_inventory,
        demand_series=demand_series,
        holding_cost=holding_cost,
        shortage_cost=shortage_cost,
        order_cost=0.5,
        return_details=True
    )

    # -------------------------
    # 4.6. Thêm thông tin nhận diện sản phẩm vào summary
    # -------------------------
    summary["id"] = product_id
    summary["item_id"] = group["item_id"].iloc[0]
    summary["dept_id"] = group["dept_id"].iloc[0]
    summary["store_id"] = group["store_id"].iloc[0]
    summary["cat_id"] = group["cat_id"].iloc[0]
    summary["state_id"] = group["state_id"].iloc[0]
    summary["unit_price"] = unit_price
    summary["holding_cost_per_unit"] = holding_cost
    summary["shortage_cost_per_unit"] = shortage_cost
    summary["estimation_window"] = f"d_{estimation_start} to d_{estimation_end}"
    summary["simulation_window"] = f"d_{simulation_start} to d_{simulation_data['d_num'].max()}"

    summary_list.append(summary)

    # -------------------------
    # 4.7. Lưu bảng chi tiết từng sản phẩm nếu cần
    # -------------------------
    results_df = results_df.copy()
    results_df["id"] = product_id
    results_df["item_id"] = group["item_id"].iloc[0]
    results_df["store_id"] = group["store_id"].iloc[0]
    results_df["d_num"] = simulation_data["d_num"].values

    details_dict[product_id] = results_df

# =========================
# 5. Tạo bảng tổng hợp summary
# =========================
summary_df = pd.DataFrame(summary_list)

# Sắp xếp lại cột cho dễ nhìn
summary_columns = [
    "id", "item_id", "dept_id", "cat_id", "store_id", "state_id",
    "unit_price", "holding_cost_per_unit", "shortage_cost_per_unit",
    "mu_demand", "std_demand",
    "review_period", "service_level_target", "z_value",
    "safety_stock", "expected_demand_over_review", "order_up_to_level",
    "initial_inventory", "num_periods_simulated", "num_orders",
    "total_demand", "total_sales", "total_shortage_units",
    "fill_rate", "cycle_service_level_empirical", "daily_service_level_empirical",
    "average_ending_inventory", "total_order_qty",
    "total_holding_cost", "total_shortage_cost", "total_order_cost", "total_cost",
    "estimation_window", "simulation_window"
]

summary_columns = [col for col in summary_columns if col in summary_df.columns]
summary_df = summary_df[summary_columns]

# =========================
# 6. Ghép toàn bộ bảng chi tiết nếu muốn
# =========================
if len(details_dict) > 0:
    details_all_df = pd.concat(details_dict.values(), ignore_index=True)
else:
    details_all_df = pd.DataFrame()

# =========================
# 7. Xem kết quả
# =========================
print(summary_df.head())
print(details_all_df.head())

                            id  item_id  dept_id  cat_id  store_id  state_id  \
0  FOODS_1_001_CA_1_evaluation     1536        4       2         0         0   
1  FOODS_1_001_CA_2_evaluation     1536        4       2         1         0   
2  FOODS_1_001_CA_3_evaluation     1536        4       2         2         0   
3  FOODS_1_001_CA_4_evaluation     1536        4       2         3         0   
4  FOODS_1_001_TX_1_evaluation     1536        4       2         4         1   

   unit_price  holding_cost_per_unit  shortage_cost_per_unit  mu_demand  \
0        5.32                 0.0532                    1.33   1.133333   
1        5.32                 0.0532                    1.33   1.133333   
2        5.32                 0.0532                    1.33   1.200000   
3        5.32                 0.0532                    1.33   0.333333   
4        5.32                 0.0532                    1.33   0.033333   

   std_demand  review_period  service_level_target   z_value  safety

In [6]:
summary_df.to_csv("../../dataset/result/simulation/simulation_summary.csv", index=False)
details_all_df.to_csv("../../dataset/result/simulation/simulation_details.csv", index=False)

## Dự báo 30 ngày trên tập train để ước lượng error của phương pháp forecast

In [11]:
#Số ngày dự báo
h = 28

#Số lượng lags tối đa
max_lags = 57

#Ngày kết thúc tập huấn luyện
tr_last = 1885

#Ngày đầu tiên của dự báo
fday = datetime(2016, 3, 28)

In [13]:
#Recursive forecasting và magic multiplier
alphas = [1.028, 1.023, 1.018]
weights = [1/len(alphas)]*len(alphas)
sub = 0.

for icount, (alpha, weight) in enumerate(zip(alphas, weights)):

    #Lấy dữ liệu
    te = create_dt(raw_folder= "../../dataset/raw/" ,is_train=False, price_dt = PRICE_DTYPES, cal_dt = CAL_DTYPES, tr_last = tr_last)

    #Add demand feature vào te
    te = add_demandFeature(demand_dir="../../dataset/result/demand_classification.csv", data = te)
    cols = [f"d_{tr_last + i}" for i in range(1,29)]

    for tdelta in range(0, 28):
        day = fday + timedelta(days=tdelta)
        print(tdelta, day)
        tst = te[(te.date >= day - timedelta(days=max_lags)) & (te.date <= day)].copy()

        create_fea(tst ,lags_fe = [7, 28], wins_fe = [7, 28])
        
        tst = tst.loc[tst.date == day , train_cols]
        te.loc[te.date == day, "sales"] = alpha*m_lgb.predict(tst) # magic multiplier by kyakovlev



    te_sub = te.loc[te.date >= fday, ["id", "sales"]].copy()

    te_sub["F"] = [f"d_{tr_last + rank}" 
               for rank in te_sub.groupby("id")["id"].cumcount() + 1]
    te_sub = te_sub.set_index(["id", "F" ]).unstack()["sales"][cols].reset_index()
    te_sub.fillna(0., inplace = True)
    te_sub.sort_values("id", inplace = True)
    te_sub.reset_index(drop=True, inplace = True)
    te_sub.to_csv(f"../../dataset/result/all/submission_sim_{icount}.csv",index=False)
    if icount == 0 :
        sub = te_sub
        sub[cols] *= weight
    else:
        sub[cols] += te_sub[cols]*weight
    print(icount, alpha, weight)


0 2016-03-28 00:00:00
1 2016-03-29 00:00:00
2 2016-03-30 00:00:00
3 2016-03-31 00:00:00
4 2016-04-01 00:00:00
5 2016-04-02 00:00:00
6 2016-04-03 00:00:00
7 2016-04-04 00:00:00
8 2016-04-05 00:00:00
9 2016-04-06 00:00:00
10 2016-04-07 00:00:00
11 2016-04-08 00:00:00
12 2016-04-09 00:00:00
13 2016-04-10 00:00:00
14 2016-04-11 00:00:00
15 2016-04-12 00:00:00
16 2016-04-13 00:00:00
17 2016-04-14 00:00:00
18 2016-04-15 00:00:00
19 2016-04-16 00:00:00
20 2016-04-17 00:00:00
21 2016-04-18 00:00:00
22 2016-04-19 00:00:00
23 2016-04-20 00:00:00
24 2016-04-21 00:00:00
25 2016-04-22 00:00:00
26 2016-04-23 00:00:00
27 2016-04-24 00:00:00
0 1.028 0.3333333333333333
0 2016-03-28 00:00:00
1 2016-03-29 00:00:00
2 2016-03-30 00:00:00
3 2016-03-31 00:00:00
4 2016-04-01 00:00:00
5 2016-04-02 00:00:00
6 2016-04-03 00:00:00
7 2016-04-04 00:00:00
8 2016-04-05 00:00:00
9 2016-04-06 00:00:00
10 2016-04-07 00:00:00
11 2016-04-08 00:00:00
12 2016-04-09 00:00:00
13 2016-04-10 00:00:00
14 2016-04-11 00:00:00
15 2

In [ ]:
#Lưu dữ liệu forecast
sub.to_csv("../../dataset/result/all/submission_sim.csv")

In [ ]:
#Load dữ liệu forecast
sub_sim = pd.read_csv("../../dataset/result/all/submission_sim.csv")
sub_sim["id"] = sub_sim["id"].str.extract(r"^(.+?)_(?:validation|evaluation)$", expand=False)
sub_sim.set_index("id", inplace = True)
sub_sim.head()

,d_1886,d_1887,d_1888,d_1889,d_1890,d_1891,d_1892,d_1893,d_1894,d_1895,d_1896,d_1897,d_1898,d_1899,d_1900,d_1901,d_1902,d_1903,d_1904,d_1905,d_1906,d_1907,d_1908,d_1909,d_1910,d_1911,d_1912,d_1913
id,,,,,,,,,,,,,,,,,,,,,,,,,,,,
FOODS_1_001_CA_1_validation,0.646279,0.629915,0.675699,0.637703,0.920313,1.071212,1.017624,0.762808,0.731810,0.742111,0.767183,0.872346,1.095887,1.081443,0.794722,0.742426,0.761833,0.811461,0.883337,1.069430,1.037118,0.692642,0.667905,0.681123,0.682363,0.821554,1.044364,1.022160
FOODS_1_001_CA_2_validation,0.848140,0.809315,0.812128,0.869256,1.227442,1.540219,1.327693,0.944624,0.971351,0.962419,1.023854,1.191966,1.425881,1.346009,0.930622,0.911665,0.933107,0.994098,1.157107,1.303172,1.271746,0.803761,0.767983,0.785275,0.831317,1.025419,1.368790,1.283804
FOODS_1_001_CA_3_validation,1.142132,1.018326,0.883858,0.880678,0.993012,1.463270,1.699619,1.314619,1.420180,1.219426,1.107898,1.251794,1.570014,1.586794,1.291932,1.241740,1.068324,1.265391,1.328864,1.545953,1.622051,1.187450,1.121113,0.947334,0.953811,1.042271,1.559519,1.536776
FOODS_1_001_CA_4_validation,0.403908,0.415354,0.455693,0.491053,0.497513,0.568645,0.520598,0.447006,0.433439,0.431639,0.470991,0.475694,0.545055,0.515618,0.416665,0.390794,0.417334,0.415861,0.433751,0.494289,0.471391,0.390714,0.384680,0.403371,0.407542,0.450810,0.526030,0.486429
FOODS_1_001_TX_1_validation,0.358458,0.361644,0.411348,0.406016,0.786535,0.682065,0.632182,0.506403,0.499346,0.505556,0.526405,0.533732,0.687645,0.624497,0.521909,0.514972,0.518779,0.521837,0.570780,0.670678,0.601943,0.460429,0.441675,0.441610,0.448293,0.527197,0.618964,0.532780


In [28]:
#Xử lý lại tập test để đánh giá
df_valid = pd.read_csv("../../dataset/raw/sales_train_evaluation.csv")
cols = ["id"] + [f"d_{i}" for i in range(tr_last + 1, 1914)]
test = df_valid[cols]
test["id"] = test["id"].str.extract(r"^(.+?)_(?:validation|evaluation)$", expand=False)

test.set_index("id", inplace = True)

#Xử lý lại tập train để đánh giá
df_valid = pd.read_csv("../../dataset/raw/sales_train_validation.csv")
df_valid["id"] = df_valid["id"].str.extract(r"^(.+?)_(?:validation|evaluation)$", expand=False)

df_valid.set_index("id", inplace = True)

# 1. Đổi đuôi id
cols_train = [f"d_{i}" for i in range(1, tr_last+ 1)]
train = df_valid[cols_train]
train.head()


#Load dữ liệu tỷ trọng
item_share = pd.read_csv("../../dataset/result/item_share.csv")
item_share.set_index("item_id", inplace = True)

In [63]:
rmsse_result = rmsse(train, test, sub_sim)
rmsse_result = rmsse_result[~np.isinf(rmsse_result)].mean()

In [64]:
wrmsse = rmsse_result * item_share['sales_share']
wrmsse.sum()

np.float64(0.7169909378073708)

## Mô phỏng tồn kho khi thêm yếu tố dự báo nhu cầu

In [5]:
test_long = create_dt(raw_folder= "../../dataset/raw/" , 
                      sales_data = 'sales_train_evaluation.csv', 
                      price_dt = PRICE_DTYPES, 
                      cal_dt = CAL_DTYPES, tr_last = 1941, first_day = 1884)

test_long["id"] = test_long["id"].str.extract(r"^(.+?)_(?:validation|evaluation)$", expand=False)
test_long.head()

,id,item_id,dept_id,store_id,cat_id,state_id,d,sales,date,wm_yr_wk,weekday,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price
0,HOBBIES_1_001_CA_1,0,0,0,0,0,d_1884,1.0,2016-03-26,11609,2,1,3,2016,0,0,0,0,0.0,0.0,0.0,8.26
1,HOBBIES_1_002_CA_1,1,0,0,0,0,d_1884,1.0,2016-03-26,11609,2,1,3,2016,0,0,0,0,0.0,0.0,0.0,3.97
2,HOBBIES_1_003_CA_1,2,0,0,0,0,d_1884,1.0,2016-03-26,11609,2,1,3,2016,0,0,0,0,0.0,0.0,0.0,2.97
3,HOBBIES_1_004_CA_1,3,0,0,0,0,d_1884,6.0,2016-03-26,11609,2,1,3,2016,0,0,0,0,0.0,0.0,0.0,4.64
4,HOBBIES_1_005_CA_1,4,0,0,0,0,d_1884,0.0,2016-03-26,11609,2,1,3,2016,0,0,0,0,0.0,0.0,0.0,2.88


In [6]:
#Load dữ liệu forecast phục vụ cho việc estimate error
sub_sim = pd.read_csv("../../dataset/result/all/submission_sim.csv")
sub_sim["id"] = sub_sim["id"].str.extract(r"^(.+?)_(?:validation|evaluation)$", expand=False)
sub_sim.head()

,id,d_1886,d_1887,d_1888,d_1889,d_1890,d_1891,d_1892,d_1893,d_1894,d_1895,d_1896,d_1897,d_1898,d_1899,d_1900,d_1901,d_1902,d_1903,d_1904,d_1905,d_1906,d_1907,d_1908,d_1909,d_1910,d_1911,d_1912,d_1913
0,FOODS_1_001_CA_1,0.646279,0.629915,0.675699,0.637703,0.920313,1.071212,1.017624,0.762808,0.731810,0.742111,0.767183,0.872346,1.095887,1.081443,0.794722,0.742426,0.761833,0.811461,0.883337,1.069430,1.037118,0.692642,0.667905,0.681123,0.682363,0.821554,1.044364,1.022160
1,FOODS_1_001_CA_2,0.848140,0.809315,0.812128,0.869256,1.227442,1.540219,1.327693,0.944624,0.971351,0.962419,1.023854,1.191966,1.425881,1.346009,0.930622,0.911665,0.933107,0.994098,1.157107,1.303172,1.271746,0.803761,0.767983,0.785275,0.831317,1.025419,1.368790,1.283804
2,FOODS_1_001_CA_3,1.142132,1.018326,0.883858,0.880678,0.993012,1.463270,1.699619,1.314619,1.420180,1.219426,1.107898,1.251794,1.570014,1.586794,1.291932,1.241740,1.068324,1.265391,1.328864,1.545953,1.622051,1.187450,1.121113,0.947334,0.953811,1.042271,1.559519,1.536776
3,FOODS_1_001_CA_4,0.403908,0.415354,0.455693,0.491053,0.497513,0.568645,0.520598,0.447006,0.433439,0.431639,0.470991,0.475694,0.545055,0.515618,0.416665,0.390794,0.417334,0.415861,0.433751,0.494289,0.471391,0.390714,0.384680,0.403371,0.407542,0.450810,0.526030,0.486429
4,FOODS_1_001_TX_1,0.358458,0.361644,0.411348,0.406016,0.786535,0.682065,0.632182,0.506403,0.499346,0.505556,0.526405,0.533732,0.687645,0.624497,0.521909,0.514972,0.518779,0.521837,0.570780,0.670678,0.601943,0.460429,0.441675,0.441610,0.448293,0.527197,0.618964,0.532780


In [7]:
#Chuyển bảng về dạng long
sub_sim_long = sub_sim.melt(
    id_vars="id",           # giữ lại id
    var_name="day",         # tên cột mới cho d_1886, d_1887,...
    value_name="forecast"   # giá trị dự báo
)

sub_sim_long = sub_sim_long.merge(test_long[['id', 'd', 'sales']], left_on=["id", "day"], right_on=["id", "d"], how="left")
sub_sim_long['error'] = sub_sim_long['sales'] - sub_sim_long['forecast']
sub_sim_long.head()

,id,day,forecast,d,sales,error
0,FOODS_1_001_CA_1,d_1886,0.646279,d_1886,2.0,1.353721
1,FOODS_1_001_CA_2,d_1886,0.848140,d_1886,0.0,-0.848140
2,FOODS_1_001_CA_3,d_1886,1.142132,d_1886,0.0,-1.142132
3,FOODS_1_001_CA_4,d_1886,0.403908,d_1886,2.0,1.596092
4,FOODS_1_001_TX_1,d_1886,0.358458,d_1886,0.0,-0.358458


In [8]:
#Load dữ liệu forecast cho 28 ngày tiếp theo để thực hiện trong mô phỏng
sub = pd.read_csv("../../dataset/result/all/submission_all.csv")
sub.drop(columns=["Unnamed: 0"], inplace=True)

sub_long = sub.melt(
    id_vars="id",           # giữ lại id
    var_name="day",         # tên cột mới cho d_1886, d_1887,...
    value_name="forecast"   # giá trị dự báo
)

sub_long

,id,day,forecast
0,FOODS_1_001_CA_1,d_1914,0.804077
1,FOODS_1_001_CA_2,d_1914,0.926463
2,FOODS_1_001_CA_3,d_1914,1.113022
3,FOODS_1_001_CA_4,d_1914,0.448898
4,FOODS_1_001_TX_1,d_1914,0.184206
...,...,...,...
853715,HOUSEHOLD_2_516_TX_2,d_1941,0.369376
853716,HOUSEHOLD_2_516_TX_3,d_1941,0.194666
853717,HOUSEHOLD_2_516_WI_1,d_1941,0.149506
853718,HOUSEHOLD_2_516_WI_2,d_1941,0.106496


In [14]:
import warnings
warnings.filterwarnings("ignore")

from IPython.display import clear_output
from scipy.stats import norm
import numpy as np
import pandas as pd

# =========================
# 1. Chuẩn bị dữ liệu
# =========================
df_sim = test_long.copy()
sub_sim_long = sub_sim_long.copy()
sub_long = sub_long.copy()

df_sim["d_num"] = df_sim["d"].str.extract(r"(\d+)", expand=False).astype("int32")
sub_sim_long["d_num"] = sub_sim_long["day"].str.extract(r"(\d+)", expand=False).astype("int32")
sub_long["d_num"] = sub_long["day"].str.extract(r"(\d+)", expand=False).astype("int32")

df_sim = df_sim.sort_values(["id", "d_num"]).reset_index(drop=True)
sub_sim_long = sub_sim_long.sort_values(["id", "d_num"]).reset_index(drop=True)
sub_long = sub_long.sort_values(["id", "d_num"]).reset_index(drop=True)

# =========================
# 2. Tham số mô phỏng
# =========================
review_period = 7
service_level = 0.95
z = norm.ppf(service_level)

estimation_start = 1884
estimation_end = 1913
simulation_start = 1914

# =========================
# 3. Cắt sẵn các vùng cần dùng
# =========================
df_est = df_sim[(df_sim["d_num"] >= estimation_start) & (df_sim["d_num"] <= estimation_end)]
df_run = df_sim[df_sim["d_num"] >= simulation_start]

err_est = sub_sim_long[
    (sub_sim_long["d_num"] >= estimation_start) & (sub_sim_long["d_num"] <= estimation_end)
]

fcst_run = sub_long[sub_long["d_num"] >= simulation_start]
fcst_first = sub_long[
    (sub_long["d_num"] > simulation_start) & (sub_long["d_num"] <= simulation_start + 7)
]

# groupby sẵn
g_est = df_est.groupby("id")
g_run = df_run.groupby("id")
g_err = err_est.groupby("id")
g_fcst_run = fcst_run.groupby("id")
g_fcst_first = fcst_first.groupby("id")

# lấy danh sách id giao nhau để tránh loop vô ích
valid_ids = (
    set(g_est.groups)
    & set(g_run.groups)
    & set(g_err.groups)
    & set(g_fcst_run.groups)
    & set(g_fcst_first.groups)
)

summary_list = []
details_list = []

for stt, product_id in enumerate(valid_ids, start=1):
    group_est = g_est.get_group(product_id)
    group_run = g_run.get_group(product_id)
    err_data = g_err.get_group(product_id)
    fcst_data = g_fcst_run.get_group(product_id)
    fcst_first_data = g_fcst_first.get_group(product_id)

    mu_error = max(err_data["error"].mean(), 0) 

    std_error = err_data["error"].std(ddof=1)
    if pd.isna(std_error):
        std_error = 0.0

    price_series = group_est["sell_price"].dropna()
    if price_series.empty:
        full_group = df_sim[df_sim["id"] == product_id]
        price_series = full_group["sell_price"].dropna()
    if price_series.empty:
        continue

    unit_price = price_series.iloc[-1]
    holding_cost = 0.01 * unit_price
    shortage_cost = 0.25 * unit_price

    expected_demand_over_review = fcst_first_data["forecast"].sum()
    safety_stock = z * std_error * np.sqrt(review_period)
    bias_correction = mu_error * review_period
    order_up_to_level = max(0, expected_demand_over_review + bias_correction + safety_stock)
    initial_inventory = order_up_to_level

    demand_series = group_run["sales"].to_numpy()
    forecast_series = fcst_data["forecast"].to_numpy()

    results_df, summary = simulate_periodic_withForecast_L0(
        forecast_series=forecast_series,
        mu_error=mu_error,
        std_error=std_error,
        review_period=review_period,
        service_level=service_level,
        initial_inventory=initial_inventory,
        demand_series=demand_series,
        holding_cost=holding_cost,
        shortage_cost=shortage_cost,
        order_cost=0.5,
        return_details=True
    )

    summary["id"] = product_id
    summary["item_id"] = group_run["item_id"].iloc[0]
    summary["dept_id"] = group_run["dept_id"].iloc[0]
    summary["store_id"] = group_run["store_id"].iloc[0]
    summary["cat_id"] = group_run["cat_id"].iloc[0]
    summary["state_id"] = group_run["state_id"].iloc[0]
    summary["unit_price"] = unit_price
    summary["holding_cost_per_unit"] = holding_cost
    summary["shortage_cost_per_unit"] = shortage_cost
    summary["estimation_window"] = f"d_{estimation_start} to d_{estimation_end}"
    summary["simulation_window"] = f"d_{simulation_start} to d_{group_run['d_num'].max()}"

    summary_list.append(summary)

    results_df = results_df.copy()
    results_df["id"] = product_id
    results_df["item_id"] = group_run["item_id"].iloc[0]
    results_df["store_id"] = group_run["store_id"].iloc[0]
    results_df["d_num"] = group_run["d_num"].to_numpy()
    details_list.append(results_df)

    # chỉ clear mỗi 100 sản phẩm
    if stt % 100 == 0:
        clear_output(wait=True)
        print(f"Processed {stt}/{len(valid_ids)} products")

summary_df = pd.DataFrame(summary_list)
details_all_df = pd.concat(details_list, ignore_index=True) if details_list else pd.DataFrame()

Processed 30400/30490 products


In [15]:
summary_df.to_csv("../../dataset/result/simulation/simulation_summary_with_forecast_error.csv")
details_all_df.to_csv("../../dataset/result/simulation/simulation_details_with_forecast_error.csv")